# NFL Big Data Bowl 2026: Fast Competitive Submission Notebook
### Compatible with `kaggle_evaluation.nfl_inference_server`
- **Dual Target Modeling**: Predicts both `x` and `y` displacement trajectories.
- **Fast 5-Fold LightGBM**: Finishes in under 3 minutes (well below the 10-minute gateway timeout).
- **Auto-FE Interactivity**: Deploys the top interaction primitives discovered in Tabular Studio.


In [ ]:
import os
import glob
import time
import warnings
import numpy as np
import pandas as pd
import polars as pl
import lightgbm as lgb
from sklearn.model_selection import GroupKFold
from sklearn.preprocessing import OrdinalEncoder

warnings.filterwarnings("ignore")
print("Environment initialized successfully.")


In [ ]:
# 1. Fast Intelligent Ingestion & Dataset Stitching
t0 = time.time()
output_files = sorted(glob.glob("/kaggle/input/**/output_*.csv", recursive=True)) or sorted(glob.glob("./**/output_*.csv", recursive=True))
input_files = sorted(glob.glob("/kaggle/input/**/input_*.csv", recursive=True)) or sorted(glob.glob("./**/input_*.csv", recursive=True))

if output_files and input_files:
    print("Stitching partition files...")
    base_df = pd.concat([pd.read_csv(f) for f in output_files], ignore_index=True)
    join_df = pd.concat([pd.read_csv(f) for f in input_files], ignore_index=True)
    p_feat = join_df.drop_duplicates(subset=["game_id", "play_id", "nfl_id"]).copy()
    p_feat = p_feat.rename(columns={"x": "x_feature", "y": "y_feature"})
    if "frame_id" in p_feat.columns:
        p_feat = p_feat.drop(columns=["frame_id"])
    train_df = pd.merge(base_df, p_feat, on=["game_id", "play_id", "nfl_id"], how="inner")
else:
    stitched = glob.glob("/kaggle/input/**/train_stitched.csv", recursive=True) or glob.glob("./**/train_stitched.csv", recursive=True)
    train_df = pd.read_csv(stitched[0])

print(f"Training matrix ready: {train_df.shape[0]} rows x {train_df.shape[1]} cols in {time.time()-t0:.1f}s")


In [ ]:
# 2. Universal Auto-FE (Interaction & Relative Displacements)
def extract_features(df):
    df = df.copy()
    # High-leverage interaction from Tabular Studio Auto-FE
    df["play_direction_x_player_side"] = df["play_direction"].astype(str) + "_" + df["player_side"].astype(str)
    df["play_direction_x_player_position"] = df["play_direction"].astype(str) + "_" + df["player_position"].astype(str)
    # Relative coordinate offsets
    if "ball_land_x" in df.columns and "x_feature" in df.columns:
        df["dist_ball_x"] = df["ball_land_x"] - df["x_feature"]
    if "ball_land_y" in df.columns and "y_feature" in df.columns:
        df["dist_ball_y"] = df["ball_land_y"] - df["y_feature"]
    return df

train_df = extract_features(train_df)
drop_cols = ["game_id", "play_id", "nfl_id", "frame_id", "x", "y", "player_name", "player_birth_date"]
feature_cols = [c for c in train_df.columns if c not in drop_cols]
cat_cols = [c for c in feature_cols if train_df[c].dtype == "object" or isinstance(train_df[c].dtype, pd.CategoricalDtype)]
print(f"Active training features: {len(feature_cols)} cols")


In [ ]:
# 3. Dual-Target Fast 5-Fold Training (Models for x and y)
enc = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)
train_df[cat_cols] = enc.fit_transform(train_df[cat_cols].astype(str))

gkf = GroupKFold(n_splits=5)
splits = list(gkf.split(train_df, train_df["x"], groups=train_df["play_id"].values))

models_x = []
models_y = []

lgb_params = {
    "n_estimators": 1200,
    "learning_rate": 0.05,
    "num_leaves": 45,
    "subsample": 0.8,
    "colsample_bytree": 0.75,
    "n_jobs": -1,
    "verbose": -1,
    "random_state": 2026
}

print("Training Dual Target LightGBM Models across 5 Folds...")
t_train_start = time.time()
for f_idx, (tr_idx, val_idx) in enumerate(splits):
    X_tr, y_tr_x, y_tr_y = train_df.iloc[tr_idx][feature_cols], train_df.iloc[tr_idx]["x"], train_df.iloc[tr_idx]["y"]
    X_va, y_va_x, y_va_y = train_df.iloc[val_idx][feature_cols], train_df.iloc[val_idx]["x"], train_df.iloc[val_idx]["y"]
    
    # Train model X
    mx = lgb.LGBMRegressor(**lgb_params)
    mx.fit(X_tr, y_tr_x, eval_set=[(X_va, y_va_x)], callbacks=[lgb.early_stopping(50, verbose=False)])
    models_x.append(mx)
    
    # Train model Y
    my = lgb.LGBMRegressor(**lgb_params)
    my.fit(X_tr, y_tr_y, eval_set=[(X_va, y_va_y)], callbacks=[lgb.early_stopping(50, verbose=False)])
    models_y.append(my)
    
    rmse_x = np.sqrt(np.mean((y_va_x - mx.predict(X_va))**2))
    rmse_y = np.sqrt(np.mean((y_va_y - my.predict(X_va))**2))
    print(f"  Fold {f_idx+1}/5 -> Val RMSE x: {rmse_x:.4f} | y: {rmse_y:.4f}")

print(f"All 10 models trained successfully in {time.time()-t_train_start:.1f}s!")


In [ ]:
# 4. Predict Function for Kaggle Inference Gateway
def predict(test: pl.DataFrame, test_input: pl.DataFrame) -> pd.DataFrame:
    test_df = test.to_pandas()
    test_in_df = test_input.to_pandas()
    
    # Extract last known feature coordinates per player
    p_feat = test_in_df.drop_duplicates(subset=["game_id", "play_id", "nfl_id"]).copy()
    p_feat = p_feat.rename(columns={"x": "x_feature", "y": "y_feature"})
    if "frame_id" in p_feat.columns:
        p_feat = p_feat.drop(columns=["frame_id"])
    
    merged = pd.merge(test_df, p_feat, on=["game_id", "play_id", "nfl_id"], how="left")
    merged = extract_features(merged)
    
    # Handle categoricals with saved encoder
    merged[cat_cols] = enc.transform(merged[cat_cols].astype(str))
    
    # Reindex columns to match training features
    X_infer = merged.reindex(columns=feature_cols).fillna(0)
    
    # Ensemble predictions across folds
    pred_x = np.mean([m.predict(X_infer) for m in models_x], axis=0)
    pred_y = np.mean([m.predict(X_infer) for m in models_y], axis=0)
    
    test_df["x"] = pred_x
    test_df["y"] = pred_y
    
    return test_df[["x", "y"]]


In [ ]:
# 5. Connect to Kaggle Inference Gateway Handshake
import kaggle_evaluation.nfl_inference_server

inference_server = kaggle_evaluation.nfl_inference_server.NFLInferenceServer(predict)

if os.getenv("KAGGLE_IS_COMPETITION_RERUN"):
    print("Production rerun mode: Serving predictions to Kaggle gateway...")
    inference_server.serve()
else:
    print("Local validation mode: Testing inference server gateway...")
    test_dir = "/kaggle/input/nfl-big-data-bowl-2026-prediction/"
    if not os.path.exists(test_dir):
        test_dir = "./nfl-big-data-bowl-2026-prediction/"
    if os.path.exists(test_dir):
        inference_server.run_local_gateway((test_dir,))
